# 05. Modelado avanzado y evaluación robusta — CRISP-DM Fases 4 y 5

Este notebook consolida la corrida oficial de Hito 3 para predecir `CONSUMO_(t+1)` por serie `DISTRIBUIDORA × UBIGEO × TARIFA`.



## 1. Configuración, datos y features

Se usa exclusivamente `data/processed/dataset_ml.parquet`. `IMPORTE` queda excluido conforme a la decisión sustentada en el EDA. Las transformaciones categóricas se ajustan dentro de cada pipeline de entrenamiento.

In [1]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

warnings.filterwarnings("ignore", category=UserWarning)

DATASET_RELATIVE = Path("data") / "processed" / "dataset_ml.parquet"
def find_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / DATASET_RELATIVE).is_file():
            return candidate
    raise FileNotFoundError(f"No se encontró {DATASET_RELATIVE}.")

ROOT = find_root()
DATASET_PATH = ROOT / DATASET_RELATIVE
OUTPUT_DIR = ROOT / "reports" / "model_outputs"
MODELS_DIR = ROOT / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

raw = pl.read_parquet(DATASET_PATH).to_pandas()
for col in ["DISTRIBUIDORA", "UBIGEO", "TARIFA"]:
    raw[col] = raw[col].astype(str)

raw["LOG_DELTA"] = np.log1p(raw["TARGET_CONSUMO_MWH"]) - np.log1p(raw["consumo_total_mwh"])
CATEGORICAL_FEATURES = ["DISTRIBUIDORA", "UBIGEO", "TARIFA"]
NUMERIC_FEATURES = [
    "MES", "consumo_total_mwh", "lag_1", "lag_2", "lag_3",
    "RATIO_LAG1", "RATIO_LAG2", "RATIO_LAG3", "ROLLING_MEAN_3",
]
raw["RATIO_LAG1"] = (raw["consumo_total_mwh"] + 1) / (raw["lag_1"] + 1)
raw["RATIO_LAG2"] = (raw["consumo_total_mwh"] + 1) / (raw["lag_2"] + 1)
raw["RATIO_LAG3"] = (raw["consumo_total_mwh"] + 1) / (raw["lag_3"] + 1)
raw["ROLLING_MEAN_3"] = (raw["consumo_total_mwh"] + raw["lag_1"] + raw["lag_2"]) / 3.0
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = "TARGET_CONSUMO_MWH"
RESIDUAL_TARGET = "LOG_DELTA"

assert "importe_total_miles" not in FEATURES
assert not raw[FEATURES + [TARGET, RESIDUAL_TARGET]].isnull().any().any()

train_df = raw.loc[raw["SPLIT"] == "TRAIN"].sort_values("PERIODO").reset_index(drop=True)
val_df = raw.loc[raw["SPLIT"] == "VALIDATION"].sort_values("PERIODO").reset_index(drop=True)
test_df = raw.loc[raw["SPLIT"] == "TEST"].sort_values("PERIODO").reset_index(drop=True)
future_df = raw.loc[raw["SPLIT"] == "FUTURE_HOLDOUT"].sort_values("PERIODO").reset_index(drop=True)
train_val_df = pd.concat([train_df, val_df], ignore_index=True).sort_values("PERIODO").reset_index(drop=True)

print(f"Dataset: {len(raw):,} filas | TRAIN={len(train_df):,} | VALIDATION={len(val_df):,} | TEST={len(test_df):,} | FUTURE_HOLDOUT={len(future_df):,}")
print("Features:", FEATURES)

Dataset: 33,174 filas | TRAIN=19,552 | VALIDATION=2,936 | TEST=3,934 | FUTURE_HOLDOUT=6,752
Features: ['DISTRIBUIDORA', 'UBIGEO', 'TARIFA', 'MES', 'consumo_total_mwh', 'lag_1', 'lag_2', 'lag_3', 'RATIO_LAG1', 'RATIO_LAG2', 'RATIO_LAG3', 'ROLLING_MEAN_3']


## 2. Validación cruzada temporal dentro de TRAIN

Los pliegues son ventanas crecientes de meses completos. No hay mezcla aleatoria ni solapamiento entre los meses de entrenamiento y validación interna.

In [2]:
def expanding_calendar_cv(frame: pd.DataFrame, n_splits: int = 3, validation_months: int = 3):
    periods = sorted(frame["PERIODO"].astype(int).unique())
    splits = []
    for fold in range(n_splits):
        validation_end = len(periods) - (n_splits - 1 - fold) * validation_months
        validation_start = validation_end - validation_months
        train_periods = periods[:validation_start]
        validation_periods = periods[validation_start:validation_end]
        train_idx = frame.index[frame["PERIODO"].astype(int).isin(train_periods)].to_numpy()
        validation_idx = frame.index[frame["PERIODO"].astype(int).isin(validation_periods)].to_numpy()
        assert max(train_periods) < min(validation_periods)
        splits.append((train_idx, validation_idx))
    return splits

cv_splits = expanding_calendar_cv(train_df)
for fold, (train_idx, validation_idx) in enumerate(cv_splits, start=1):
    print(
        f"Fold {fold}: train {train_df.loc[train_idx, 'PERIODO'].min()}–{train_df.loc[train_idx, 'PERIODO'].max()} "
        f"({len(train_idx):,}) | validación {train_df.loc[validation_idx, 'PERIODO'].min()}–{train_df.loc[validation_idx, 'PERIODO'].max()} ({len(validation_idx):,})"
    )

Fold 1: train 202303–202402 (10,444) | validación 202403–202405 (2,729)
Fold 2: train 202303–202405 (13,173) | validación 202406–202408 (2,678)
Fold 3: train 202303–202408 (15,851) | validación 202409–202411 (3,701)


## 3. Tuning dentro de TRAIN



In [3]:
def make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ("numericas", "passthrough", NUMERIC_FEATURES),
    ])

def make_pipeline(model) -> Pipeline:
    return Pipeline([( "preprocesamiento", make_preprocessor()), ("modelo", model)])

BASELINE_NUMERIC_FEATURES = ["MES", "consumo_total_mwh", "lag_1", "lag_2", "lag_3"]
BASELINE_FEATURES = CATEGORICAL_FEATURES + BASELINE_NUMERIC_FEATURES
def make_baseline_tree_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ("numericas", "passthrough", BASELINE_NUMERIC_FEATURES),
    ])
    tree = DecisionTreeRegressor(criterion="squared_error", max_depth=8, min_samples_leaf=10, random_state=RANDOM_STATE)
    return Pipeline([("preprocesamiento", preprocessor), ("modelo", tree)])

search_specs = {
    "LightGBM": (
        make_pipeline(lgb.LGBMRegressor(objective="regression_l1", random_state=RANDOM_STATE, n_jobs=2, verbosity=-1)),
        {
            "modelo__learning_rate": [0.03, 0.05, 0.08],
            "modelo__num_leaves": [15, 31, 63],
            "modelo__min_child_samples": [20, 35, 50],
            "modelo__n_estimators": [80, 120, 160],
            "modelo__subsample": [0.8],
            "modelo__colsample_bytree": [0.8],
        },
    ),
    "XGBoost": (
        make_pipeline(xgb.XGBRegressor(objective="reg:absoluteerror", random_state=RANDOM_STATE, n_jobs=2, tree_method="hist")),
        {
            "modelo__learning_rate": [0.03, 0.05, 0.08],
            "modelo__max_depth": [5, 6, 7],
            "modelo__min_child_weight": [1, 5],
            "modelo__n_estimators": [80, 120, 160],
            "modelo__subsample": [0.8],
            "modelo__colsample_bytree": [0.8],
        },
    ),
    "CatBoost": (
        make_pipeline(cb.CatBoostRegressor(loss_function="MAE", random_seed=RANDOM_STATE, verbose=0, thread_count=2, allow_writing_files=False)),
        {
            "modelo__learning_rate": [0.03, 0.05, 0.08],
            "modelo__depth": [5, 6, 7],
            "modelo__l2_leaf_reg": [1, 3, 5],
            "modelo__iterations": [80, 120, 160],
        },
    ),
}

searches, selected_params = {}, {}
X_train = train_df[FEATURES]
y_train_residual = train_df[RESIDUAL_TARGET]
for name, (pipeline, params) in search_specs.items():
    search = RandomizedSearchCV(
        pipeline, params, n_iter=8, scoring="neg_mean_absolute_error", cv=cv_splits,
        random_state=RANDOM_STATE, n_jobs=1, refit=True,
    )
    search.fit(X_train, y_train_residual)
    searches[name] = search
    selected_params[name] = search.best_params_
    results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
    results.to_csv(OUTPUT_DIR / f"resultados_randomsearch_{name.lower().replace('lightgbm', 'lgbm').replace('xgboost', 'xgb').replace('catboost', 'cb')}.csv", index=False)
    print(f"{name}: {search.best_params_}")

LightGBM: {'modelo__subsample': 0.8, 'modelo__num_leaves': 31, 'modelo__n_estimators': 120, 'modelo__min_child_samples': 50, 'modelo__learning_rate': 0.03, 'modelo__colsample_bytree': 0.8}
XGBoost: {'modelo__subsample': 0.8, 'modelo__n_estimators': 80, 'modelo__min_child_weight': 1, 'modelo__max_depth': 7, 'modelo__learning_rate': 0.03, 'modelo__colsample_bytree': 0.8}
CatBoost: {'modelo__learning_rate': 0.05, 'modelo__l2_leaf_reg': 5, 'modelo__iterations': 120, 'modelo__depth': 7}


## 4. Validation y selección oficial del campeón



In [4]:
def metrics(y_true, prediction, reference_mae=None):
    mae = mean_absolute_error(y_true, prediction)
    result = {
        "MAE_MWh": float(mae),
        "RMSE_MWh": float(np.sqrt(mean_squared_error(y_true, prediction))),
        "R2": float(r2_score(y_true, prediction)),
        "WAPE_pct": float(np.abs(y_true - prediction).sum() / np.abs(y_true).sum() * 100),
    }
    result["MEJORA_MAE_pct"] = None if reference_mae is None else float((reference_mae - mae) / reference_mae * 100)
    return result

def restore_level(frame, predicted_delta):
    return np.clip(np.expm1(np.log1p(frame["consumo_total_mwh"].to_numpy()) + predicted_delta), 0, None)

X_val = val_df[FEATURES]
y_val = val_df[TARGET].to_numpy()
p_val_ref = val_df["consumo_total_mwh"].to_numpy()
reference_val = metrics(y_val, p_val_ref)

baseline_pipeline = make_baseline_tree_pipeline()
baseline_pipeline.fit(train_df[BASELINE_FEATURES], train_df[TARGET])
p_val_tree = baseline_pipeline.predict(val_df[BASELINE_FEATURES])

val_predictions = {"Referencia Temporal (Persistencia)": p_val_ref, "Decision Tree Baseline (Hito 2)": p_val_tree}
for name, search in searches.items():
    val_predictions[f"{name} Avanzado (Hito 3)"] = restore_level(val_df, search.best_estimator_.predict(X_val))

param_label = {
    "Referencia Temporal (Persistencia)": "No aplica",
    "Decision Tree Baseline (Hito 2)": json.dumps({"max_depth": 8, "min_samples_leaf": 10, "criterion": "squared_error"}, ensure_ascii=False),
    **{f"{name} Avanzado (Hito 3)": json.dumps(params, ensure_ascii=False, sort_keys=True) for name, params in selected_params.items()},
}
val_rows = []
for model_name, prediction in val_predictions.items():
    val_rows.append({"MODELO": model_name, **metrics(y_val, prediction, reference_val["MAE_MWh"]), "HIPERPARAMETROS_SELECCIONADOS": param_label[model_name]})
df_val_metrics = pd.DataFrame(val_rows).sort_values("MAE_MWh").reset_index(drop=True)
df_val_metrics.to_csv(OUTPUT_DIR / "metricas_hito3_validation.csv", index=False)
champion_row = df_val_metrics.loc[df_val_metrics["MODELO"].str.contains("Avanzado")].iloc[0]
CHAMPION_MODEL = champion_row["MODELO"]
CHAMPION_FAMILY = CHAMPION_MODEL.split(" Avanzado")[0]
print("=== SELECCIÓN OFICIAL EN VALIDATION ===")
print(df_val_metrics[["MODELO", "MAE_MWh", "RMSE_MWh", "R2", "WAPE_pct", "MEJORA_MAE_pct"]].to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\nCampeón oficial: {CHAMPION_MODEL}; seleccionado por menor MAE en VALIDATION ({champion_row['MAE_MWh']:.6f} MWh).")

=== SELECCIÓN OFICIAL EN VALIDATION ===
                            MODELO   MAE_MWh   RMSE_MWh       R2  WAPE_pct  MEJORA_MAE_pct
        LightGBM Avanzado (Hito 3) 11.355295  70.968586 0.994730  6.030123       22.473146
         XGBoost Avanzado (Hito 3) 12.037300  71.353055 0.994672  6.392295       17.816844
        CatBoost Avanzado (Hito 3) 12.813041  81.022329 0.993131  6.804245       12.520570
Referencia Temporal (Persistencia) 14.646919  87.544245 0.991980  7.778109        0.000000
   Decision Tree Baseline (Hito 2) 17.423408 102.283172 0.989052  9.252538      -18.956133

Campeón oficial: LightGBM Avanzado (Hito 3); seleccionado por menor MAE en VALIDATION (11.355295 MWh).


## 5. Reentrenamiento y evaluación final en TEST

Los tres ensambles se reentrenan con TRAIN + VALIDATION usando los hiperparámetros fijados. La tabla permite comparar el desempeño final, pero no modifica el campeón ya seleccionado con Validation.

In [5]:
X_train_val = train_val_df[FEATURES]
y_train_val_residual = train_val_df[RESIDUAL_TARGET]
X_test = test_df[FEATURES]
y_test = test_df[TARGET].to_numpy()
p_test_ref = test_df["consumo_total_mwh"].to_numpy()
reference_test = metrics(y_test, p_test_ref)

final_models = {}
for name, (pipeline, _) in search_specs.items():
    pipeline.set_params(**selected_params[name])
    pipeline.fit(X_train_val, y_train_val_residual)
    final_models[name] = pipeline

final_tree = make_baseline_tree_pipeline()
final_tree.fit(train_val_df[BASELINE_FEATURES], train_val_df[TARGET])

p_test_tree = final_tree.predict(test_df[BASELINE_FEATURES])
test_predictions = {
    "Referencia Temporal (Persistencia)": p_test_ref,
    "Decision Tree Baseline (Hito 2)": p_test_tree,
}
for name, model in final_models.items():
    test_predictions[f"{name} Avanzado (Hito 3)"] = restore_level(test_df, model.predict(X_test))

test_rows = []
for model_name, prediction in test_predictions.items():
    test_rows.append({"MODELO": model_name, **metrics(y_test, prediction, reference_test["MAE_MWh"]), "SELECCIONADO_EN_VALIDATION": model_name == CHAMPION_MODEL})
df_test_metrics = pd.DataFrame(test_rows).sort_values("MAE_MWh").reset_index(drop=True)
df_test_metrics.to_csv(OUTPUT_DIR / "metricas_hito3_comparativa.csv", index=False)

print("=== EVALUACIÓN FINAL EN TEST ===")
print(df_test_metrics[["MODELO", "MAE_MWh", "RMSE_MWh", "R2", "WAPE_pct", "MEJORA_MAE_pct", "SELECCIONADO_EN_VALIDATION"]].to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\nRegla aplicada: {CHAMPION_MODEL} se mantiene como campeón porque fue elegido en VALIDATION; TEST no cambia esta elección.")

=== EVALUACIÓN FINAL EN TEST ===
                            MODELO   MAE_MWh  RMSE_MWh       R2  WAPE_pct  MEJORA_MAE_pct  SELECCIONADO_EN_VALIDATION
        LightGBM Avanzado (Hito 3)  8.670241 48.845157 0.997327  4.751691       14.948672                        True
         XGBoost Avanzado (Hito 3)  8.690999 48.998403 0.997310  4.763067       14.745053                       False
        CatBoost Avanzado (Hito 3)  9.145190 53.841833 0.996752  5.011985       10.289628                       False
Referencia Temporal (Persistencia) 10.194128 58.970655 0.996104  5.586852        0.000000                       False
   Decision Tree Baseline (Hito 2) 12.715930 66.128861 0.995101  6.968915      -24.737787                       False

Regla aplicada: LightGBM Avanzado (Hito 3) se mantiene como campeón porque fue elegido en VALIDATION; TEST no cambia esta elección.


## 6. Predicciones oficiales y análisis de errores del modelo campeón

El análisis usa exclusivamente TEST. Residuo positivo significa subpredicción; residuo negativo, sobrepredicción. Los rangos de consumo se construyen con cuartiles de `TARGET_REAL` en TEST.

In [6]:
test_eval = test_df[["DISTRIBUIDORA", "UBIGEO", "TARIFA", "PERIODO", "TARGET_PERIODO"]].copy()
test_eval["TARGET_REAL"] = y_test
test_eval["PRED_REFERENCIA"] = p_test_ref
test_eval["PRED_DECISION_TREE"] = p_test_tree
test_eval["PRED_LIGHTGBM"] = test_predictions["LightGBM Avanzado (Hito 3)"]
test_eval["PRED_XGBOOST"] = test_predictions["XGBoost Avanzado (Hito 3)"]
test_eval["PRED_CATBOOST"] = test_predictions["CatBoost Avanzado (Hito 3)"]
champion_prediction_column = {"LightGBM": "PRED_LIGHTGBM", "XGBoost": "PRED_XGBOOST", "CatBoost": "PRED_CATBOOST"}[CHAMPION_FAMILY]
test_eval["PRED_CAMPEON"] = test_eval[champion_prediction_column]
test_eval["MODELO_CAMPEON"] = CHAMPION_MODEL
test_eval["RESIDUO"] = test_eval["TARGET_REAL"] - test_eval["PRED_CAMPEON"]
test_eval["ERROR_ABSOLUTO"] = test_eval["RESIDUO"].abs()
test_eval["ERROR_PORCENTUAL"] = np.where(test_eval["TARGET_REAL"] != 0, test_eval["ERROR_ABSOLUTO"] / test_eval["TARGET_REAL"] * 100, np.nan)
test_eval.to_csv(OUTPUT_DIR / "predicciones_test_comparativa.csv", index=False)

residual = test_eval["RESIDUO"]
error_summary = {
    "modelo_campeon": CHAMPION_MODEL,
    "observaciones_test": int(len(test_eval)),
    "media_residuo": float(residual.mean()),
    "mediana_residuo": float(residual.median()),
    "desviacion_residuo": float(residual.std(ddof=1)),
    "p1_residuo": float(residual.quantile(0.01)),
    "p5_residuo": float(residual.quantile(0.05)),
    "p95_residuo": float(residual.quantile(0.95)),
    "p99_residuo": float(residual.quantile(0.99)),
    "pct_subprediccion": float((residual > 0).mean() * 100),
    "pct_sobreprediccion": float((residual < 0).mean() * 100),
    "media_subprediccion": float(residual[residual > 0].mean()),
    "media_sobreprediccion": float((-residual[residual < 0]).mean()),
}

def grouped_error(frame: pd.DataFrame, group_cols):
    rows = []
    for keys, sub in frame.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row.update({
            "OBSERVACIONES": len(sub),
            "MAE_MWh": mean_absolute_error(sub["TARGET_REAL"], sub["PRED_CAMPEON"]),
            "RMSE_MWh": float(np.sqrt(mean_squared_error(sub["TARGET_REAL"], sub["PRED_CAMPEON"]))),
            "R2": r2_score(sub["TARGET_REAL"], sub["PRED_CAMPEON"]) if len(sub) > 1 else np.nan,
            "WAPE_pct": float(sub["ERROR_ABSOLUTO"].sum() / sub["TARGET_REAL"].abs().sum() * 100) if sub["TARGET_REAL"].abs().sum() else np.nan,
            "RESIDUO_MEDIO": float(sub["RESIDUO"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows)

errors_dist = grouped_error(test_eval, ["DISTRIBUIDORA"]).sort_values("DISTRIBUIDORA")
errors_tariff = grouped_error(test_eval, ["TARIFA"]).sort_values("OBSERVACIONES", ascending=False)
errors_ubigeo = grouped_error(test_eval, ["DISTRIBUIDORA", "UBIGEO"]).sort_values("MAE_MWh", ascending=False)
errors_month = grouped_error(test_eval, ["TARGET_PERIODO"]).sort_values("TARGET_PERIODO")

q = test_eval["TARGET_REAL"].quantile([0, .25, .50, .75, 1]).to_numpy()
q = np.unique(q)
labels = ["Bajo", "Medio", "Alto", "Muy alto"][:len(q)-1]
test_eval["RANGO_CONSUMO_REAL"] = pd.cut(test_eval["TARGET_REAL"], bins=q, labels=labels, include_lowest=True, duplicates="drop")
errors_range = grouped_error(test_eval, ["RANGO_CONSUMO_REAL"]).sort_values("RANGO_CONSUMO_REAL")
errors_range["LIMITE_INFERIOR_MWh"] = q[:-1]
errors_range["LIMITE_SUPERIOR_MWh"] = q[1:]

worst_predictions = test_eval.sort_values("ERROR_ABSOLUTO", ascending=False).head(20)[[
    "DISTRIBUIDORA", "UBIGEO", "TARIFA", "PERIODO", "TARGET_PERIODO", "TARGET_REAL", "PRED_CAMPEON", "ERROR_ABSOLUTO", "ERROR_PORCENTUAL"
]].copy()

errors_dist.to_csv(OUTPUT_DIR / "hito3_errores_por_distribuidora.csv", index=False)
errors_tariff.to_csv(OUTPUT_DIR / "hito3_errores_por_tarifa.csv", index=False)
errors_ubigeo.to_csv(OUTPUT_DIR / "hito3_errores_por_ubigeo.csv", index=False)
errors_month.to_csv(OUTPUT_DIR / "hito3_errores_por_mes.csv", index=False)
errors_range.to_csv(OUTPUT_DIR / "hito3_errores_por_rango_consumo.csv", index=False)
worst_predictions.to_csv(OUTPUT_DIR / "hito3_peores_predicciones.csv", index=False)
errors_dist.to_csv(OUTPUT_DIR / "metricas_hito3_por_distribuidora.csv", index=False)
errors_tariff.to_csv(OUTPUT_DIR / "metricas_hito3_por_tarifa.csv", index=False)
with open(OUTPUT_DIR / "hito3_resumen_errores_campeon.json", "w", encoding="utf-8") as file:
    json.dump(error_summary, file, ensure_ascii=False, indent=2)

print("=== RESUMEN DE RESIDUOS DEL CAMPEÓN EN TEST ===")
print(json.dumps(error_summary, ensure_ascii=False, indent=2))
print("\nPor distribuidora:")
print(errors_dist.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nTop 20 peores predicciones:")
print(worst_predictions.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

=== RESUMEN DE RESIDUOS DEL CAMPEÓN EN TEST ===
{
  "modelo_campeon": "LightGBM Avanzado (Hito 3)",
  "observaciones_test": 3934,
  "media_residuo": -2.2119928851752753,
  "mediana_residuo": 3.2417924145100585e-07,
  "desviacion_residuo": 48.801247802421436,
  "p1_residuo": -103.74036533436411,
  "p5_residuo": -13.112926543332502,
  "p95_residuo": 10.408534211632933,
  "p99_residuo": 72.57562814791115,
  "pct_subprediccion": 50.355871886120994,
  "pct_sobreprediccion": 48.83070665988816,
  "media_subprediccion": 6.412607150747817,
  "media_sobreprediccion": 11.142818727699613
}

Por distribuidora:
DISTRIBUIDORA  OBSERVACIONES  MAE_MWh  RMSE_MWh     R2  WAPE_pct  RESIDUO_MEDIO
        ENOSA           2072   9.8400   48.5932 0.9956    6.4378        -2.7948
   HIDRANDINA           1862   7.3686   49.1240 0.9981    3.4204        -1.5635

Top 20 peores predicciones:
DISTRIBUIDORA UBIGEO TARIFA  PERIODO  TARGET_PERIODO  TARGET_REAL  PRED_CAMPEON  ERROR_ABSOLUTO  ERROR_PORCENTUAL
   HIDRANDIN

## 7. Visualizaciones del error

Las visualizaciones se limitan a distribución de residuos, real frente a predicho, MAE mensual y MAE por rango de consumo. Los outliers se conservan y se reportan en la tabla de peores predicciones.

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(test_eval["RESIDUO"], bins=60, kde=True, ax=axes[0, 0], color="#2563eb")
axes[0, 0].axvline(0, color="#dc2626", linestyle="--")
axes[0, 0].set_title(f"Residuos — {CHAMPION_FAMILY} (TEST)")
axes[0, 0].set_xlabel("Real - predicho (MWh)")

axes[0, 1].scatter(np.log1p(test_eval["TARGET_REAL"]), np.log1p(test_eval["PRED_CAMPEON"]), alpha=.25, s=10)
limit = max(axes[0, 1].get_xlim()[1], axes[0, 1].get_ylim()[1])
axes[0, 1].plot([0, limit], [0, limit], "r--")
axes[0, 1].set_title("Real vs predicho (log1p)")
axes[0, 1].set_xlabel("log1p(real)")
axes[0, 1].set_ylabel("log1p(predicho)")

axes[1, 0].plot(errors_month["TARGET_PERIODO"].astype(str), errors_month["MAE_MWh"], marker="o")
axes[1, 0].set_title("MAE por TARGET_PERIODO")
axes[1, 0].set_ylabel("MAE (MWh)")
axes[1, 0].tick_params(axis="x", rotation=30)

axes[1, 1].bar(errors_range["RANGO_CONSUMO_REAL"].astype(str), errors_range["MAE_MWh"], color="#10b981")
axes[1, 1].set_title("MAE por rango de consumo real")
axes[1, 1].set_ylabel("MAE (MWh)")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "hito3_analisis_errores_campeon.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Persistencia, manifiesto y estado de FUTURE_HOLDOUT

Se persiste el pipeline completo del campeón y se valida su recarga. El manifiesto registra el protocolo y los outputs de esta corrida. FUTURE_HOLDOUT no intervino en esta ejecución; su evaluación histórica previa se identifica como evaluación temporal extendida, no como parte de la selección.

In [8]:
champion_pipeline = final_models[CHAMPION_FAMILY]
champion_model_path = MODELS_DIR / "modelo_campeon_hito3.joblib"
joblib.dump(champion_pipeline, champion_model_path)
if CHAMPION_FAMILY == "LightGBM":
    joblib.dump(champion_pipeline, MODELS_DIR / "lightgbm_hito3.joblib")
reloaded_champion = joblib.load(champion_model_path)
assert np.allclose(reloaded_champion.predict(X_test), champion_pipeline.predict(X_test))

manifest = {
    "run_name": "hito3_oficial_validation_selection_v1",
    "executed_at": datetime.now().astimezone().isoformat(),
    "dataset": str(DATASET_PATH.relative_to(ROOT)),
    "target": TARGET,
    "internal_training_target": RESIDUAL_TARGET,
    "features": FEATURES,
    "random_state": RANDOM_STATE,
    "splits": {
        "TRAIN": "TARGET_PERIODO <= 202412",
        "VALIDATION": "202501 <= TARGET_PERIODO <= 202503",
        "TEST": "202504 <= TARGET_PERIODO <= 202507",
        "FUTURE_HOLDOUT": "TARGET_PERIODO > 202507",
    },
    "cv": {"type": "expanding calendar window", "folds": 3, "validation_months": 3, "source": "TRAIN only"},
    "tuning_metric": "negative MAE on LOG_DELTA inside TRAIN CV",
    "selection_metric": "MAE_MWh on VALIDATION",
    "champion": {"name": CHAMPION_MODEL, "family": CHAMPION_FAMILY, "hyperparameters": selected_params[CHAMPION_FAMILY], "model_path": str(champion_model_path.relative_to(ROOT))},
    "validation_metrics": df_val_metrics.to_dict(orient="records"),
    "test_metrics": df_test_metrics.to_dict(orient="records"),
    "future_holdout": "No participó en entrenamiento, tuning, selección ni análisis de errores de esta corrida. Fue evaluado previamente como evaluación temporal extendida y sus artefactos históricos no son outputs oficiales de esta corrida.",
    "obsolete_or_nonofficial_artifacts": [
        "predicciones_evaluacion_completa_2026.csv", "matriz_completa_estrategias_algoritmos.csv",
        "comparativa_estrategias_arquitectura.csv", "desglose_27_modelos_arquitectura.csv",
        "resultados_gridsearch_lgbm.csv", "predicciones_test_lgbm.csv"
    ],
    "outputs": [
        "metricas_hito3_validation.csv", "metricas_hito3_comparativa.csv", "predicciones_test_comparativa.csv",
        "resultados_randomsearch_lgbm.csv", "resultados_randomsearch_xgb.csv", "resultados_randomsearch_cb.csv",
        "hito3_errores_por_distribuidora.csv", "hito3_errores_por_tarifa.csv", "hito3_errores_por_ubigeo.csv",
        "hito3_errores_por_mes.csv", "hito3_errores_por_rango_consumo.csv", "hito3_peores_predicciones.csv",
        "hito3_resumen_errores_campeon.json", "hito3_analisis_errores_campeon.png"
    ],
}
with open(OUTPUT_DIR / "hito3_run_manifest.json", "w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)
print(f"Modelo campeón guardado y recargado: {champion_model_path}")
print(f"Manifiesto: {OUTPUT_DIR / 'hito3_run_manifest.json'}")

Modelo campeón guardado y recargado: C:\Users\rafae\Downloads\Hito2_1ACC0057_4773_GRUPO_08\models\modelo_campeon_hito3.joblib
Manifiesto: C:\Users\rafae\Downloads\Hito2_1ACC0057_4773_GRUPO_08\reports\model_outputs\hito3_run_manifest.json


## 9. Integración con base de datos común

Nos decidimos por integrar una base de datos PostgreSQL local.

La arquitectura implementada es:

**Modelo LightGBM → resultados oficiales → PostgreSQL → Dashboard Streamlit**

La base de datos `ml_consumo` funciona como fuente común para almacenar los resultados producidos por el módulo de IA.

Se implementaron tres tablas principales:

- `model_runs`: almacena información de cada ejecución del modelo, configuración y métricas globales.
- `predictions`: almacena las predicciones individuales realizadas sobre el conjunto TEST.
- `segment_metrics`: almacena métricas de evaluación por distribuidora, tarifa, UBIGEO, periodo y rango de consumo.

La escritura de resultados se realiza mediante `scripts/persist_hito3_results.py`, mientras que `scripts/verify_database.py` permite comprobar la integridad de los registros almacenados.

In [2]:
import os
import psycopg
from dotenv import load_dotenv

load_dotenv("../.env")

conn = psycopg.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            model_name,
            test_mae,
            test_rmse,
            test_r2,
            test_wape
        FROM model_runs
        ORDER BY created_at DESC
        LIMIT 1;
    """)

    resultado = cur.fetchone()

conn.close()

print("Modelo almacenado:", resultado[0])
print(f"MAE TEST:  {resultado[1]:.4f} MWh")
print(f"RMSE TEST: {resultado[2]:.4f} MWh")
print(f"R² TEST:   {resultado[3]:.4f}")
print(f"WAPE TEST: {resultado[4]:.4f}%")

Modelo almacenado: LightGBM Avanzado (Hito 3)
MAE TEST:  8.6702 MWh
RMSE TEST: 48.8452 MWh
R² TEST:   0.9973
WAPE TEST: 4.7517%


## 10. Conclusiones del Hito 3

Se evaluaron LightGBM, XGBoost y CatBoost utilizando ajuste de hiperparámetros y validación cruzada temporal. La selección del modelo se realizó exclusivamente mediante VALIDATION, donde LightGBM obtuvo el menor MAE y fue seleccionado como modelo campeón.

Posteriormente, el modelo fue reentrenado y evaluado sobre TEST, obteniendo aproximadamente:

- MAE: **8.67 MWh**
- RMSE: **48.85 MWh**
- R²: **0.9973**
- WAPE: **4.75 %**

LightGBM superó tanto a la referencia de persistencia como al Decision Tree desarrollado durante el Hito 2.

El análisis de errores permitió estudiar el desempeño por distribuidora, tarifa, UBIGEO, periodo y rango de consumo, además de identificar las observaciones con mayores errores y posibles valores extremos.

Finalmente, los resultados oficiales del módulo de IA fueron persistidos en una base de datos PostgreSQL local. La base almacena la corrida del modelo, las 3,934 predicciones de TEST y las métricas segmentadas, y funciona como fuente común para el dashboard predictivo.